# Real checkpoint files: what CV-Pilot verifies, refuses and resumes

The first notebooks used checkpoints that CV-Pilot wrote itself. This one uses the files that
training frameworks write: a **PyTorch Lightning** `.ckpt`, an **Ultralytics** YOLO11n `last.pt`, a
**Hugging Face Trainer** checkpoint folder and an **MMEngine** `.pth`, all produced here by real
(short, CPU) training runs. For each of them it asks the questions a user cares about:

1. Can the strict reader take the file at all, and if not, what exactly does it say?
2. If it can: does `cvpilot ckpt verify` pass, is a flipped byte, a truncation and a swapped-in
   foreign file refused, does `resume_path` hand back the file, and does the framework end up with
   **the same weights** as a run that was never interrupted?
3. How much does a save cost on real model sizes, and where does the time go?

Every number below is computed from the run that printed it. Lines that start with `CHECK:` are
computed pass/fail results, and the last cell lists every file layout with the exact message the
reader gave. Where a result is a refusal or a limitation, it is shown as such.

Data and models are the ones of the earlier notebooks (COCO128: annotations CC BY 4.0, the images
keep their Flickr licences; YOLO11n through the `ultralytics` package, AGPL-3.0; `microsoft/resnet-18`
at a pinned commit, Apache-2.0). Downloads are SHA-256 pinned and checked before anything is
unpacked or loaded. **Files that training writes are loaded by the framework that wrote them in this
notebook only after CV-Pilot has accepted them or, for the Ultralytics and Hugging Face native
resume controls, because this notebook produced them moments before.**

In [1]:
import contextlib
import hashlib
import json
import os
import re
import shlex
import shutil
import socket
import subprocess
import sys
import tempfile
import time
import urllib.error
import urllib.request
import warnings
import zipfile
from pathlib import Path

warnings.filterwarnings("ignore")

# Where downloads are cached. This is outside the repository and nothing in it is ever committed.
DATA = Path(
    os.environ.get("CVPILOT_NOTEBOOK_DATA") or Path(tempfile.gettempdir()) / "cvpilot-notebooks"
)
DATA.mkdir(parents=True, exist_ok=True)
# A fresh scratch folder for this run of the notebook: everything the notebook writes lives here.
REPO = Path.cwd()  # the notebook is run from the repository root (see notebooks/README.md)
WORK = Path(tempfile.mkdtemp(prefix="cvpilot-nb-"))
os.chdir(WORK)

os.environ["CVPILOT_STORE_DIR"] = "store"  # runs, metrics and evaluation results
os.environ["CVPILOT_CHECKPOINT_DIR"] = "ckpt"  # checkpoints
os.environ["CVPILOT_PROJECT"] = "real-files"  # default project of every cvpilot command
os.environ["CVPILOT_CKPT_ALLOW_PT"] = "1"  # framework files are stored in the opt-in pt format
os.environ["CVPILOT_LOG_LEVEL"] = "WARNING"  # keeps this notebook's own Python calls quiet
os.environ["YOLO_VERBOSE"] = "False"  # silence the progress bars of Ultralytics
os.environ["YOLO_CONFIG_DIR"] = str(DATA / "ultralytics-config")  # keep Ultralytics settings here
(DATA / "ultralytics-config").mkdir(exist_ok=True)
os.environ.pop("CVPILOT_ENV", None)  # production behaviour until a cell says otherwise
os.environ.update(
    HF_HUB_OFFLINE="1",
    TRANSFORMERS_VERBOSITY="error",
    TOKENIZERS_PARALLELISM="false",
    PYTHONWARNINGS="ignore",
    HF_HUB_DISABLE_TELEMETRY="1",
    TQDM_DISABLE="1",
    HF_HUB_DISABLE_PROGRESS_BARS="1",
)  # the training processes never touch the network
TIMINGS = {}  # seconds per step, printed at the end

print("python", sys.version.split()[0], "| scratch folder: a fresh temporary directory")
print("downloads are cached in $CVPILOT_NOTEBOOK_DATA (default: <tmp>/cvpilot-notebooks)")

python 3.13.9 | scratch folder: a fresh temporary directory
downloads are cached in $CVPILOT_NOTEBOOK_DATA (default: <tmp>/cvpilot-notebooks)


### Getting ready

The same helpers as the earlier notebooks: a fresh scratch folder, `sh()` to run a command and show
its exit code, `scrub()` to hide this computer's paths, and the pinned, checked downloads.

In [2]:
_REPLACEMENTS = sorted(
    [
        (str(WORK.resolve()), "."),
        (str(WORK), "."),
        (str(DATA.resolve()), "$CVPILOT_NOTEBOOK_DATA"),
        (str(DATA), "$CVPILOT_NOTEBOOK_DATA"),
        (str(Path.home()), "~"),
        (socket.gethostname(), "<this-computer>"),
    ],
    key=lambda pair: -len(pair[0]),
)
_OTHER_PATHS = re.compile(r"/(?:Users|home|private|var|tmp|opt)/[^\s\"',:)]+")
_QUIET_FIELDS = {"logger", "request_id", "service", "env", "@timestamp", "level", "message"}
_CLI = "import sys; sys.argv[0] = 'cvpilot'; from cvpilot.cli.app import main; main()"


def scrub(text):
    """Hide the paths and the computer name of the machine that runs this notebook."""
    for old, new in _REPLACEMENTS:
        if old:
            text = text.replace(old, new)
    return _OTHER_PATHS.sub("<path>", text)


def log_summary(line):
    """cvpilot writes one JSON object per log line on stderr; show it as one short line."""
    try:
        event = json.loads(line)
    except ValueError:
        return "stderr: " + scrub(line)
    if not isinstance(event, dict):
        return "stderr: " + scrub(line)
    fields = " ".join(
        f"{key}={scrub(str(value))[:44]}"
        for key, value in event.items()
        if key not in _QUIET_FIELDS
    )
    return f"log[{event.get('level', '?')}] {event.get('message')} {fields}"


class Result:
    def __init__(self, code, out, err):
        self.code, self.out, self.err = code, out, err

    def json(self):
        return json.loads(self.out)


def sh(
    command, *, lines=40, log_lines=8, quiet=False, keep=False, env=None, expect=None, timeout=900
):
    """Run one command, print what it printed (paths hidden) and its exit code.

    `cvpilot ...` runs the command-line tool of this Python environment. Output that is not
    JSON on stderr is shown as it is; the JSON log lines are shortened to one line each.
    The result (exit code, output) is returned only with `keep=True` or `quiet=True`.
    """
    argv = shlex.split(command)
    if argv[0] == "cvpilot":
        argv = [sys.executable, "-c", _CLI, *argv[1:]]
    elif argv[0] == "python":
        argv[0] = sys.executable
    full_env = {**os.environ, "CVPILOT_LOG_LEVEL": "INFO", **(env or {})}
    proc = subprocess.run(  # noqa: S603
        argv, capture_output=True, text=True, cwd=WORK, env=full_env, timeout=timeout, check=False
    )
    result = Result(proc.returncode, scrub(proc.stdout), scrub(proc.stderr))
    if not quiet:
        print("$", scrub(command))
        out_lines = result.out.rstrip("\n").splitlines()
        for line in out_lines[:lines]:
            print(line[:180] + (" ..." if len(line) > 180 else ""))
        if len(out_lines) > lines:
            print(f"... ({len(out_lines) - lines} more lines)")
        err_lines = [ln for ln in proc.stderr.splitlines() if ln.strip()]
        for line in err_lines[:log_lines]:
            print(log_summary(line)[:200])
        if len(err_lines) > log_lines:
            print(f"... ({len(err_lines) - log_lines} more log lines)")
        print("exit code:", proc.returncode)
    if expect is not None and proc.returncode != expect:
        raise RuntimeError(f"expected exit code {expect}, got {proc.returncode}: {command}")
    return result if (keep or quiet) else None


@contextlib.contextmanager
def timed(label):
    start = time.perf_counter()
    yield
    TIMINGS[label] = round(time.perf_counter() - start, 1)
    print(f"[{label}: {TIMINGS[label]} s]")

In [3]:
CHECKS = []  # (label, passed): printed as CHECK lines and counted at the end
LAYOUTS = []  # one row per real file layout: framework, file, outcome, exact message


def check(label, passed):
    CHECKS.append((label, bool(passed)))
    print(f"CHECK: {label}: {bool(passed)}")


def layout(framework, file, outcome, message=""):
    LAYOUTS.append((framework, file, outcome, scrub(message)))

### Checking what we download

Exactly the checked fetch of the earlier notebooks: https only, a size limit and a timeout, a temporary
file renamed only after its SHA-256 matched the pin, and a zip validator (no absolute paths, no `..`,
no links, bounded size and ratio) instead of `extractall`. A `.pt` file is a pickle: the YOLO11n weights
are loaded only after the digest check passed.

In [4]:
COCO128_URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip"
WEIGHTS_URL = "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt"
# Pinned SHA-256 of every download. A file with any other digest is deleted and the notebook stops.
PINNED_SHA256 = {
    "coco128.zip": "61e5e3028863d8ffc3b81d6a514603954889f0edd5e4b44c4ce60b2da99aeb8e",
    "yolo11n.pt": "0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1",
}
MAX_DOWNLOAD_BYTES = {"coco128.zip": 20_000_000, "yolo11n.pt": 20_000_000}
ZIP_LIMITS = {"files": 1000, "bytes": 50_000_000, "ratio": 20}


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def fetch(url, name):
    """Download over https into the cache (bounded, via a temporary file) and check the SHA-256.

    A cached file with the wrong digest is deleted and fetched again; a download with the wrong
    digest is deleted and raises. Nothing is unpacked or loaded before this check passes."""
    dest = DATA / name
    if dest.exists() and sha256_of(dest) != PINNED_SHA256[name]:
        dest.unlink()  # never reuse a cached file that does not match
    if not dest.exists():
        if not url.startswith("https://"):
            raise ValueError("only https downloads are allowed")
        part = dest.with_name(name + ".part")
        try:
            with urllib.request.urlopen(url, timeout=60) as response, open(part, "wb") as out:  # noqa: S310
                total = 0
                while chunk := response.read(1 << 20):
                    total += len(chunk)
                    if total > MAX_DOWNLOAD_BYTES[name]:
                        raise RuntimeError(
                            f"{name} is larger than {MAX_DOWNLOAD_BYTES[name]} bytes"
                        )
                    out.write(chunk)
        except (urllib.error.URLError, TimeoutError, OSError) as exc:
            part.unlink(missing_ok=True)
            raise RuntimeError(
                f"Could not download {name} ({type(exc).__name__}). This notebook needs the "
                f"internet once. Download {url} yourself, put the file in the folder named by "
                "CVPILOT_NOTEBOOK_DATA (default: <tmp>/cvpilot-notebooks) and run this cell again."
            ) from None
        except RuntimeError:
            part.unlink(missing_ok=True)
            raise
        digest = sha256_of(part)
        if digest != PINNED_SHA256[name]:
            part.unlink()
            raise RuntimeError(f"{name}: SHA-256 {digest} is not the pinned {PINNED_SHA256[name]}")
        part.replace(dest)  # atomic: a half-downloaded file never has the final name
    print(
        f"{name:12s} {dest.stat().st_size / 1e6:5.1f} MB  sha256 {sha256_of(dest)}  (matches the pin)"
    )
    return dest


def safe_unzip(zip_path, target):
    """Unpack into a new folder after checking every member; never `extractall`."""
    if target.exists():
        return
    staging = Path(tempfile.mkdtemp(dir=DATA, prefix="unzip-"))
    with zipfile.ZipFile(zip_path) as archive:
        members = archive.infolist()
        too_big = sum(m.file_size for m in members) > ZIP_LIMITS["bytes"]
        if len(members) > ZIP_LIMITS["files"] or too_big:
            raise RuntimeError("the archive has too many files or is too large")
        for member in members:
            name = Path(member.filename)
            mode = member.external_attr >> 16
            if name.is_absolute() or ".." in name.parts or "\\" in member.filename:
                raise RuntimeError(f"unsafe name in the archive: {member.filename}")
            if mode and (mode & 0o170000) not in (0o100000, 0o040000):
                raise RuntimeError(f"not a plain file or folder: {member.filename}")
            ratio = member.file_size / max(member.compress_size, 1)
            if ratio > ZIP_LIMITS["ratio"]:
                raise RuntimeError(f"suspicious compression ratio: {member.filename}")
        for member in members:
            out = staging / member.filename
            if member.is_dir():
                out.mkdir(parents=True, exist_ok=True)
                continue
            out.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as src, open(out, "wb") as dst:
                shutil.copyfileobj(src, dst)
    staging.rename(target)


coco_zip = fetch(COCO128_URL, "coco128.zip")
weights_path = fetch(WEIGHTS_URL, "yolo11n.pt")  # a pickle: only loaded after the check above
COCO_DIR = DATA / "coco128-unpacked"
safe_unzip(coco_zip, COCO_DIR)
COCO_ROOT = COCO_DIR / "coco128"
IMAGES = sorted((COCO_ROOT / "images" / "train2017").glob("*.jpg"))
print(f"{len(IMAGES)} images in $CVPILOT_NOTEBOOK_DATA/coco128-unpacked/coco128/images/train2017")

coco128.zip    7.0 MB  sha256 61e5e3028863d8ffc3b81d6a514603954889f0edd5e4b44c4ce60b2da99aeb8e  (matches the pin)
yolo11n.pt     5.6 MB  sha256 0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1  (matches the pin)
128 images in $CVPILOT_NOTEBOOK_DATA/coco128-unpacked/coco128/images/train2017


In [5]:
HF_REPO = "microsoft/resnet-18"
HF_REVISION = "65a5785d9156231087c481e0c7dd33a5ff6f7e3e"  # a commit id, not a branch name
HF_BASE = f"https://huggingface.co/{HF_REPO}/resolve/{HF_REVISION}"
# SHA-256 pins. The weights' pin is the Hub's own lfs.oid for this commit; the config's was
# computed from the downloaded bytes and cross-checked against the Hub's git blob id below.
PINNED_SHA256.update(
    {
        "resnet18-model.safetensors": "1cf00ee468998c23d084361b02cffadabacb5074105564c596b8079f77ecb126",
        "resnet18-config.json": "11a01bcc873444bb433140e5f30f3687356641ed923266d738eef1c3438f6d22",
    }
)
MAX_DOWNLOAD_BYTES.update(
    {"resnet18-model.safetensors": 60_000_000, "resnet18-config.json": 200_000}
)
GIT_BLOB_ID = {
    "resnet18-config.json": "be9ec1cb232f596559a0c780dedcf3d42fe11613"
}  # from the Hub listing

hf_weights = fetch(f"{HF_BASE}/model.safetensors", "resnet18-model.safetensors")
hf_config = fetch(f"{HF_BASE}/config.json", "resnet18-config.json")
config_bytes = hf_config.read_bytes()
blob = hashlib.sha1(b"blob %d\0" % len(config_bytes) + config_bytes).hexdigest()  # noqa: S324
if blob != GIT_BLOB_ID["resnet18-config.json"]:
    raise RuntimeError("config.json differs from the Hub's git blob id")
print("config.json git blob id matches the Hub listing:", blob[:12] + "...")

HF_MODEL_DIR = DATA / "resnet18-pinned"
if not HF_MODEL_DIR.exists():
    staging = Path(tempfile.mkdtemp(dir=DATA, prefix="hf-"))
    shutil.copyfile(hf_weights, staging / "model.safetensors")
    shutil.copyfile(hf_config, staging / "config.json")
    staging.rename(HF_MODEL_DIR)
from safetensors import safe_open

with safe_open(str(HF_MODEL_DIR / "model.safetensors"), framework="np") as handle:
    print(
        f"model.safetensors holds {len(list(handle.keys()))} tensors and no code (safetensors format)"
    )
os.environ["HF_MODEL_DIR"] = str(HF_MODEL_DIR)
INJECT_STEP = 8  # the step at which the NaN is injected (the training script is told the same)

resnet18-model.safetensors  46.8 MB  sha256 1cf00ee468998c23d084361b02cffadabacb5074105564c596b8079f77ecb126  (matches the pin)
resnet18-config.json   0.1 MB  sha256 11a01bcc873444bb433140e5f30f3687356641ed923266d738eef1c3438f6d22  (matches the pin)
config.json git blob id matches the Hub listing: be9ec1cb232f...
model.safetensors holds 122 tensors and no code (safetensors format)


In [6]:
import numpy as np
import torch

with timed("import the frameworks"):
    import lightning
    import mmengine
    import transformers
    import ultralytics
    from lightning.pytorch.callbacks import ModelCheckpoint

    from cvpilot.ckpt import manager as ckpt_manager
    from cvpilot.ckpt import serializers

    ultralytics.settings.update({"sync": False})  # no usage reporting from this notebook
import logging

for logger_name in ("lightning", "lightning.pytorch", "lightning.fabric", "torch.distributed"):
    logging.getLogger(logger_name).setLevel(
        logging.ERROR
    )  # keep this notebook's output to its own lines
import cvpilot
from cvpilot.ckpt import CheckpointManager
from cvpilot.hooks import lightning as lightning_hook

print(
    f"torch {torch.__version__}, lightning {lightning.__version__}, ultralytics {ultralytics.__version__}, "
    f"transformers {transformers.__version__}, mmengine {mmengine.__version__}"
)
print(
    f"cvpilot {cvpilot.__version__}; the strict reader needs torch >= "
    f"{'.'.join(map(str, serializers.MIN_PT_VERSION))} (installed: {torch.__version__})"
)

[import the frameworks: 2.9 s]
torch 2.14.1, lightning 2.6.6, ultralytics 8.4.177, transformers 5.20.1, mmengine 0.10.7
cvpilot 0.1.0; the strict reader needs torch >= 2.6 (installed: 2.14.1)


**What this shows.** The notebook runs against the real framework releases printed above. The strict
checkpoint reader is only willing to hand a `.pt` file to a framework whose `weights_only=True` loader is
safe, which is why the installed version is printed.

# Part A. Real data: a small image set from COCO128

The training runs need something to learn from. This cell turns the COCO128 photos into small
colour thumbnails whose label is the most frequent object class of the photo (only the most common
classes are kept). It is a toy task on purpose: the point of this notebook is the checkpoint files,
not the accuracy.

In [7]:
from collections import Counter

from PIL import Image

NAMES = [ultralytics.YOLO(str(weights_path)).names[i] for i in range(80)]  # the 80 COCO class names


def photo_label(image):
    label_file = COCO_ROOT / "labels" / "train2017" / f"{image.stem}.txt"
    if not label_file.exists():
        return None
    classes = [int(ln.split()[0]) for ln in label_file.read_text().splitlines() if ln.strip()]
    return Counter(classes).most_common(1)[0][0] if classes else None


photo_classes = {image.stem: photo_label(image) for image in IMAGES}
frequent = [
    c for c, _ in Counter(v for v in photo_classes.values() if v is not None).most_common(5)
]
ids = [image for image in IMAGES if photo_classes[image.stem] in frequent]
X = (
    np.stack(
        [
            np.asarray(Image.open(p).convert("RGB").resize((16, 16)), dtype=np.float32) / 255
            for p in ids
        ]
    )
    .transpose(0, 3, 1, 2)
    .reshape(len(ids), -1)
)
Y = np.array([frequent.index(photo_classes[p.stem]) for p in ids])
print(f"{len(ids)} photos, {X.shape[1]} features each, classes: {[NAMES[c] for c in frequent]}")
print(
    "photos per class:",
    dict(zip([NAMES[c] for c in frequent], np.bincount(Y).tolist(), strict=True)),
)

60 photos, 768 features each, classes: ['person', 'dog', 'car', 'cup', 'giraffe']
photos per class: {'person': 41, 'dog': 6, 'car': 6, 'cup': 4, 'giraffe': 3}


**What this shows.** Real photos, real labels, a task small enough to train in seconds on a CPU.

# Part B. PyTorch Lightning: a real `.ckpt`

A tiny linear classifier is trained for two epochs without interruption (the reference), and again for
one epoch with the CV-Pilot hook attached (the run that gets "killed"). The file Lightning's
`ModelCheckpoint(save_last=True)` wrote is then adopted, verified, tampered with and finally used to
resume.

In [8]:
class Tiny(lightning.LightningModule):
    def __init__(self):
        super().__init__()
        self.layer = torch.nn.Linear(X.shape[1], len(frequent))

    def training_step(self, batch, _index):
        x, y = batch
        loss = torch.nn.functional.cross_entropy(self.layer(x), y)
        self.log("loss", loss)
        return loss

    def configure_optimizers(self):
        return torch.optim.SGD(self.parameters(), lr=0.5, momentum=0.9)


def loader():
    data = torch.utils.data.TensorDataset(torch.tensor(X), torch.tensor(Y))
    return torch.utils.data.DataLoader(data, batch_size=16, shuffle=False)


def fit(epochs, folder, *, ckpt_path=None, attach=None):
    """Train for `epochs` epochs (counted from the start, also when resuming)."""
    lightning.seed_everything(0, verbose=False)
    saver = ModelCheckpoint(dirpath=folder, save_last=True)
    trainer = lightning.Trainer(
        max_epochs=epochs,
        callbacks=[saver],
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
        accelerator="cpu",
        log_every_n_steps=1,
    )
    run = attach(trainer) if attach else None
    model = Tiny()
    trainer.fit(model, loader(), ckpt_path=ckpt_path)
    return model, saver, trainer, run


def same_weights(a, b):
    pairs = zip(a.state_dict().values(), b.state_dict().values(), strict=True)
    return all(torch.equal(x, y) for x, y in pairs)


with timed("Lightning: two epochs straight"):
    straight, straight_saver, _, _ = fit(2, WORK / "light-straight")
with timed("Lightning: one epoch, hook attached"):
    half, half_saver, half_trainer, half_run = fit(
        1,
        WORK / "light-half",
        attach=lambda t: cvpilot.autolog(t, project="light", task="classification", name="tiny"),
    )
cvpilot.finish("preempted")  # stands in for the kill: the job did not get to the second epoch
last_file = Path(half_saver.last_model_path)
print(
    f"Lightning wrote {last_file.name}: {last_file.stat().st_size} bytes at global step {half_trainer.global_step}"
)
from cvpilot.store.local import open_store


def recorded(project, run_id, key_part):
    """The points of one run whose key contains `key_part`, read back from the store."""
    store = open_store(project, Path("store"))
    try:
        return [p for p in store.read_points(run_id) if key_part in p.key]
    finally:
        store.close()


loss_points = recorded("light", half_run.run_id, "loss")
check(
    "the hook recorded one loss value per real training step, in step order",
    [p.step for p in loss_points] == sorted({p.step for p in loss_points})
    and len(loss_points) == half_trainer.global_step,
)

[Lightning: two epochs straight: 0.0 s]


[Lightning: one epoch, hook attached: 0.2 s]
Lightning wrote last.ckpt: 35545 bytes at global step 4
CHECK: the hook recorded one loss value per real training step, in step order: True


**What this shows.** Two real Lightning runs. The hook logged the real `self.log("loss")` values while
the run was alive and the run was marked `preempted` as the job stopped. `last.ckpt` is the file the
framework itself would resume from.

### What does the file look like, and can the strict reader take it?

`describe()` reads only the zip headers and runs the same scan `resume_path` runs (`scan_pt_archive`:
zip structure, member names, sizes, compression ratio, and an opcode-level scan of the pickle against an
allow-list of tensor types). Nothing is unpickled.

In [9]:
def describe(path):
    data = Path(path).read_bytes()
    info = {
        "bytes": len(data),
        "members": 0,
        "descriptor_flag": 0,
        "zip64_end": b"PK\x06\x06" in data[-200:],
    }
    try:
        with zipfile.ZipFile(path) as archive:
            members = archive.infolist()
            info["members"] = len(members)
            info["descriptor_flag"] = sum(1 for m in members if m.flag_bits & 0x08)
    except zipfile.BadZipFile:
        info["members"] = None
    try:
        info["scan"] = ("accepted", serializers.scan_pt_archive(data), "")
    except Exception as exc:
        info["scan"] = ("refused", None, f"{getattr(exc, 'reason', type(exc).__name__)}: {exc}")
    return info


def show(framework, label, path):
    info = describe(path)
    outcome, tensor_bytes, message = info["scan"]
    print(
        f"{label}: {info['bytes']} bytes, {info['members']} zip members, "
        f"{info['descriptor_flag']} with the data-descriptor flag, zip64 end record: {info['zip64_end']}"
    )
    print(
        f"   strict scan: {outcome}"
        + (
            f" ({tensor_bytes} bytes of tensor data)"
            if tensor_bytes is not None
            else f" -> {message}"
        )
    )
    layout(framework, label, outcome, message)
    return outcome


check(
    "Lightning last.ckpt is accepted by the strict reader",
    show("Lightning", "last.ckpt", last_file) == "accepted",
)

last.ckpt: 35545 bytes, 11 zip members, 11 with the data-descriptor flag, zip64 end record: True
   strict scan: accepted (30764 bytes of tensor data)
CHECK: Lightning last.ckpt is accepted by the strict reader: True


**What this shows.** The framework's writer marks every member with the zip data-descriptor flag and puts
a zip64 end record in front of the classic one, even for a small file. A reader that only knew the
layout Python's own `zipfile` produces refuses every file the framework writes; this one accepts exactly
this layout (and the plain one) and nothing looser.

### Adopt, verify, resume

`adopt` brings the framework's own file under CV-Pilot's verification as a `pt` checkpoint (manifest, SHA-256,
digest in the run store). `cvpilot ckpt verify` is the command line check. `resume_path` is what a restarted
script calls; it returns a verified path, and the Lightning helper turns it into `ckpt_path=`.

In [10]:
run = cvpilot.init("light", task="classification", name="adopt-step", resume_from=half_run.run_id)
mgr = CheckpointManager(run, dir=WORK / "ckpt", allow_pt_format=True, install_signal_handlers=False)
manifest = mgr.adopt(last_file, step=half_trainer.global_step)
mgr.close()
cvpilot.finish("preempted")
print(
    f"adopted: format {manifest.format}, step {manifest.step}, {manifest.size_bytes} bytes, sha256 {manifest.sha256[:16]}..."
)
sh("cvpilot ckpt verify ckpt --project light --require-trust anchored", log_lines=3)

adopted: format pt, step 4, 35545 bytes, sha256 8cdbd6fe5d16f02b...


$ cvpilot ckpt verify ckpt --project light --require-trust anchored
01M56C43K7QFCV7X0KTBZH609M  last  000000004  valid    anchored
log[INFO] checkpoint_verified trust_level=anchored run_id=01M56C43K7QFCV7X0KTBZH609M kind=last step=4
exit code: 0


In [11]:
point = cvpilot.resume_path("light", with_trust=True)
args = lightning_hook.resume_arguments(point)
print(
    "resume point: step",
    point.manifest.step,
    "| trust",
    point.trust,
    "| ckpt_path given to Lightning is the verified file:",
    Path(args["ckpt_path"]).name,
)
resumed, _, resumed_trainer, _ = fit(2, WORK / "light-resumed", ckpt_path=args["ckpt_path"])
check(
    "a resumed Lightning run ends with the same weights as the uninterrupted run",
    same_weights(straight, resumed),
)
check(
    "the resumed run continued from the adopted step, not from zero",
    resumed_trainer.global_step == 2 * half_trainer.global_step,
)
check(
    "the verified file is byte-identical to the file Lightning wrote",
    point.path.read_bytes() == last_file.read_bytes(),
)

resume point: step 4 | trust anchored | ckpt_path given to Lightning is the verified file: step_000000004.pt
CHECK: a resumed Lightning run ends with the same weights as the uninterrupted run: True
CHECK: the resumed run continued from the adopted step, not from zero: True
CHECK: the verified file is byte-identical to the file Lightning wrote: True


**What this shows.** The adopted file verified, the helper handed Lightning only that verified path, and
the resumed run reached the same weights as the run that was never interrupted (bitwise equal, CPU, one
seed). The comparison is computed, not asserted by hand.

### Tamper: flip a byte, truncate, swap in a foreign file

The stored copy is modified on disk, then checked three ways: `cvpilot ckpt verify` (exit code),
`resume_path` under the default policy and under `anchored` (the run store knows the step existed).
The "foreign" file is a **valid** Lightning checkpoint from a different training run: it is
well-formed, so only the recorded digest can tell it apart.

In [12]:
stored = WORK / "ckpt" / manifest.run_id / manifest.path
good = stored.read_bytes()
foreign = Path(straight_saver.last_model_path).read_bytes()
attacks = {
    "flip one byte": good[:100] + bytes([good[100] ^ 1]) + good[101:],
    "truncate to half": good[: len(good) // 2],
    "swap in a foreign checkpoint": foreign,
}


PROBES = {
    "probe_resume.py": """
import json
import cvpilot

try:
    found = cvpilot.resume_path("light")
    print(json.dumps({"returned": found is not None}))
except Exception as exc:
    print(json.dumps({"raised": type(exc).__name__, "reason": getattr(exc, "reason", "")}))
""",
    "probe_restore.py": """
import json
import os
import cvpilot
from cvpilot.ckpt import CheckpointManager

run = cvpilot.init("hf", task="classification", name="probe", resume_from=os.environ["PROBE_PARENT"])
manager = CheckpointManager(run, dir=os.environ["PROBE_DIR"], install_signal_handlers=False)
try:
    print(json.dumps({"returned": manager.restore() is not None}))
except Exception as exc:
    print(json.dumps({"raised": type(exc).__name__, "reason": getattr(exc, "reason", "")}))
""",
}
for name, text in PROBES.items():
    Path(name).write_text(text)


def run_probe(script, env=None):
    """What a restarted script does, in its own process: its answer, and the warnings it logged."""
    result = sh(f"python {script}", quiet=True, env=env or {})
    answer = json.loads(result.out.strip().splitlines()[-1])
    events = []
    for line in result.err.splitlines():
        try:
            event = json.loads(line)
        except ValueError:
            continue
        if event.get("level") == "WARN":
            detail = event.get("reason") or event.get("reasons") or ""
            events.append(f"{event['message']}({detail})" if detail else event["message"])
    shown = (
        "a file was returned"
        if answer.get("returned")
        else (
            "None (nothing handed back)"
            if "raised" not in answer
            else f"{answer['raised']} ({answer['reason']})"
        )
    )
    return shown, events


def outcome_of(policy):
    return run_probe("probe_resume.py", {"CVPILOT_CKPT_REQUIRE_TRUST": policy})


TAMPER_ROWS = []
for name, bad in attacks.items():
    stored.chmod(0o600)
    stored.write_bytes(bad)
    verify = sh("cvpilot ckpt verify ckpt --project light", quiet=True)
    (default, default_events), (anchored, _) = outcome_of("any"), outcome_of("anchored")
    row = (name, verify.code, default, anchored)
    TAMPER_ROWS.append(row)
    print(f"{name:30s} verify exit {row[1]} | resume_path default: {row[2]} | anchored: {row[3]}")
    print(f"{'':30s} warnings logged: {', '.join(dict.fromkeys(default_events))}")
stored.write_bytes(good)
stored.chmod(0o444)
print(
    f"{'restored original':30s} verify exit {sh('cvpilot ckpt verify ckpt --project light', quiet=True).code}"
)
check(
    "every tampered Lightning file made `ckpt verify` exit 1", all(r[1] == 1 for r in TAMPER_ROWS)
)
check(
    "no tampered file was handed back by resume_path under the default or the anchored policy",
    all("a file was returned" not in r[2] + r[3] for r in TAMPER_ROWS),
)
check(
    "the original verifies again after the file is put back",
    sh("cvpilot ckpt verify ckpt --project light", quiet=True).code == 0,
)

flip one byte                  verify exit 1 | resume_path default: None (nothing handed back) | anchored: CheckpointUntrustedError (checkpoint_missing)
                               warnings logged: checkpoint_skipped_corrupt(digest_mismatch), checkpoint_progress_lost(rolled_back), resume_point_unavailable(['digest_mismatch'])


truncate to half               verify exit 1 | resume_path default: None (nothing handed back) | anchored: CheckpointUntrustedError (checkpoint_missing)
                               warnings logged: checkpoint_skipped_corrupt(size_mismatch), checkpoint_progress_lost(rolled_back), resume_point_unavailable(['size_mismatch'])


swap in a foreign checkpoint   verify exit 1 | resume_path default: None (nothing handed back) | anchored: CheckpointUntrustedError (checkpoint_missing)
                               warnings logged: checkpoint_skipped_corrupt(digest_mismatch), checkpoint_progress_lost(rolled_back), resume_point_unavailable(['digest_mismatch'])


restored original              verify exit 0
CHECK: every tampered Lightning file made `ckpt verify` exit 1: True
CHECK: no tampered file was handed back by resume_path under the default or the anchored policy: True


CHECK: the original verifies again after the file is put back: True


**What this shows.** All three modifications are caught by the digest, including the swap with a perfectly
valid foreign checkpoint. Under the default policy the file is simply not returned (a warning event says
why); under `anchored` the same situation raises, because the run store proves that step existed and
silently starting from scratch would hide lost progress.

### The same real file under a key (`authenticated`)

The strongest trust level proves who wrote a checkpoint, with an HMAC tag made from a key that only the
environment holds. The key below is random for this run and is never printed. The same Lightning file is
adopted into a new run with the key set, then verified under the `authenticated` policy: with the right
key, with another key, with no key, and after a byte of the stored file is flipped.

In [13]:
import secrets

KEY, OTHER_KEY = secrets.token_hex(32), secrets.token_hex(32)  # random, per run, never printed
os.environ["CVPILOT_CKPT_KEY"] = KEY
run = cvpilot.init("light-signed", task="classification", name="signed")
mgr = CheckpointManager(
    run, dir=WORK / "ckpt-signed", allow_pt_format=True, install_signal_handlers=False
)
signed = mgr.adopt(last_file, step=half_trainer.global_step)
mgr.close()
cvpilot.finish("preempted")
os.environ.pop("CVPILOT_CKPT_KEY")
print(
    f"adopted under a key: manifest tagged with algorithm {signed.tag_alg}, key id {signed.key_id[:8]}..."
)
VERIFY = "cvpilot ckpt verify ckpt-signed --project light-signed --require-trust authenticated"
right = sh(VERIFY, env={"CVPILOT_CKPT_KEY": KEY}, log_lines=2, keep=True)
other = sh(VERIFY, env={"CVPILOT_CKPT_KEY": OTHER_KEY}, log_lines=2, keep=True)
nokey = sh(VERIFY, log_lines=2, keep=True)
stored = WORK / "ckpt-signed" / signed.run_id / signed.path
good = stored.read_bytes()
stored.chmod(0o600)
stored.write_bytes(good[:100] + bytes([good[100] ^ 1]) + good[101:])
flipped = sh(VERIFY, env={"CVPILOT_CKPT_KEY": KEY}, log_lines=2, keep=True)
stored.write_bytes(good)
stored.chmod(0o444)
check(
    "the right key verifies the real file at the authenticated level",
    right.code == 0 and "authenticated" in right.out,
)
check("another key does not", other.code != 0)
check("no key under the authenticated policy does not", nokey.code != 0)
check("a flipped byte does not, even with the right key", flipped.code != 0)
check(
    "the key never appears in any output",
    not any(
        KEY in r.out + r.err or OTHER_KEY in r.out + r.err for r in (right, other, nokey, flipped)
    ),
)

adopted under a key: manifest tagged with algorithm hmac-sha256-v1, key id a589ef44...


$ cvpilot ckpt verify ckpt-signed --project light-signed --require-trust authenticated
01M56C46ZAVB5MF0RZ98VXR607  last  000000004  valid    authenticated
log[INFO] checkpoint_verified trust_level=authenticated run_id=01M56C46ZAVB5MF0RZ98VXR607 kind=last step=4
exit code: 0


$ cvpilot ckpt verify ckpt-signed --project light-signed --require-trust authenticated
01M56C46ZAVB5MF0RZ98VXR607  last  000000004  corrupt  key_changed
log[INFO] checkpoint_unverified_tags_ignored run_id=01M56C46ZAVB5MF0RZ98VXR607 kind=last count=1 has_key=True
log[WARN] checkpoint_untrusted reason=key_changed run_id=01M56C46ZAVB5MF0RZ98VXR607 kind=last step=4
exit code: 1


$ cvpilot ckpt verify ckpt-signed --project light-signed --require-trust authenticated
stderr: error: require_trust=authenticated needs the key in CVPILOT_CKPT_KEY (never stored)
exit code: 2


$ cvpilot ckpt verify ckpt-signed --project light-signed --require-trust authenticated
01M56C46ZAVB5MF0RZ98VXR607  last  000000004  corrupt  digest_mismatch
log[WARN] checkpoint_verify_failed reason=digest_mismatch run_id=01M56C46ZAVB5MF0RZ98VXR607 kind=last step=4
exit code: 1
CHECK: the right key verifies the real file at the authenticated level: True
CHECK: another key does not: True
CHECK: no key under the authenticated policy does not: True
CHECK: a flipped byte does not, even with the right key: True
CHECK: the key never appears in any output: True


**What this shows.** A real framework file can be brought under the strongest level: the tag binds the
manifest to the key, so another key or no key is refused, and the digest still catches a modified file
under the right key. Exit code `1` means a check failed and `2` means the command was used wrongly (the
no-key case: the policy cannot be met), as in the earlier notebooks.

### Hostile files: a pickle that would run a command

Two real files written by the framework's own `save`: a model object (what Ultralytics writes) and an
object whose pickle would call `os.system`. They are never loaded. `adopt` scans them first and stores
nothing.

In [14]:
import os as _os


class WouldRunACommand:
    def __reduce__(self):
        return (_os.system, ("echo this must never run",))


hostile = {
    "model object": {"model": torch.nn.Linear(2, 2), "epoch": 1},
    "os.system call": {"x": WouldRunACommand()},
}
run = cvpilot.init("hostile", task="classification", name="hostile")
mgr = CheckpointManager(
    run, dir=WORK / "ckpt-hostile", allow_pt_format=True, install_signal_handlers=False
)
for name, obj in hostile.items():
    path = WORK / f"hostile-{name.split()[0]}.pt"
    torch.save(obj, path)
    try:
        mgr.adopt(path, step=1)
        result = "ADOPTED"
    except Exception as exc:
        result = f"{type(exc).__name__}[{getattr(exc, 'reason', '')}]: {exc}"
    print(f"{name:16s} -> {scrub(result)}")
    layout(
        "synthetic hostile",
        f"{name} saved by the framework",
        "refused" if result != "ADOPTED" else "ADOPTED",
        result,
    )
check("adopt refused both hostile files", all(r[2] == "refused" for r in LAYOUTS[-2:]))
check("nothing was stored for them", mgr.list() == [])
mgr.close()
cvpilot.finish("failed")

model object     -> CheckpointCorruptError[unsafe_archive]: the pickle refers to torch.nn.modules.linear.Linear
os.system call   -> CheckpointCorruptError[unsafe_archive]: the pickle refers to posix.system
CHECK: adopt refused both hostile files: True
CHECK: nothing was stored for them: True


**What this shows.** The scan names the global the pickle refers to and stops before anything executes.
The refusal happens when the file is adopted, so a file that resume would never accept is not stored
and reported as saved.

# Part C. Ultralytics YOLO11n: a real `last.pt`

YOLO11n (the pinned COCO weights) is trained on COCO128 for two epochs at a reduced image size on the
CPU. To get a file that needs a resume, the first run is stopped by a callback that raises right after
the first epoch was saved. The CV-Pilot hook is attached to every run.

In [15]:
YAML = WORK / "coco128.yaml"
YAML.write_text(
    f"path: {COCO_ROOT}\ntrain: images/train2017\nval: images/train2017\nnames:\n"
    + "".join(f"  {i}: {n}\n" for i, n in enumerate(NAMES))
)
TRAIN = dict(
    data=str(YAML),
    epochs=2,
    imgsz=320,
    batch=16,
    workers=0,
    device="cpu",
    plots=False,
    amp=False,
    seed=0,
    deterministic=True,
    verbose=False,
    exist_ok=True,
    project=str(WORK / "yolo"),
)


class SimulatedKill(Exception):
    pass


def kill_after_first_save(trainer):
    raise SimulatedKill("stopped after the first epoch was saved")


def results(name):
    import csv

    with open(WORK / "yolo" / name / "results.csv") as handle:
        rows = list(csv.DictReader(handle))
    return [{k.strip(): float(v) for k, v in row.items()} for row in rows]


with timed("YOLO11n: two epochs straight"):
    ultralytics.YOLO(str(weights_path)).train(name="straight", **TRAIN)
model = ultralytics.YOLO(str(weights_path))
yolo_run = cvpilot.autolog(model, project="yolo", task="detection", name="interrupted")
model.add_callback("on_model_save", kill_after_first_save)
with timed("YOLO11n: first epoch, then the kill"):
    try:
        model.train(name="interrupted", **TRAIN)
    except SimulatedKill as exc:
        print("training stopped:", exc)
yolo_last = WORK / "yolo" / "interrupted" / "weights" / "last.pt"
status = cvpilot.runs.get_run_view("yolo", yolo_run.run_id).meta.status
print(
    f"last.pt: {yolo_last.stat().st_size} bytes; epochs in results.csv: {len(results('interrupted'))}; CV-Pilot run status: {status}"
)
check("the hook marked the killed YOLO run failed", status == "failed")

[YOLO11n: two epochs straight: 15.4 s]


training stopped: stopped after the first epoch was saved
[YOLO11n: first epoch, then the kill: 6.8 s]
last.pt: 21471971 bytes; epochs in results.csv: 1; CV-Pilot run status: failed
CHECK: the hook marked the killed YOLO run failed: True


**What this shows.** A real Ultralytics training run and a real interruption. The hook recorded the epoch
metrics before the stop and marked the run failed instead of leaving it `running`.

### Can CV-Pilot verify and resume this file?

The same scan, applied to the file Ultralytics wrote and to the pinned pretrained weights.

In [16]:
check(
    "YOLO last.pt is refused by the strict reader",
    show("Ultralytics", "last.pt (training state, written by train)", yolo_last) == "refused",
)
show("Ultralytics", "yolo11n.pt (pretrained weights, the pinned download)", weights_path)
run = cvpilot.init("yolo", task="detection", name="adopt-yolo")
mgr = CheckpointManager(
    run, dir=WORK / "ckpt-yolo", allow_pt_format=True, install_signal_handlers=False
)
try:
    mgr.adopt(yolo_last, step=1)
    adopted = True
except Exception as exc:
    adopted = False
    print("adopt:", type(exc).__name__, f"[{getattr(exc, 'reason', '')}]:", scrub(str(exc)))
check("adopt stored nothing for last.pt", (not adopted) and mgr.list() == [])
mgr.close()
cvpilot.finish("failed")

last.pt (training state, written by train): 21471971 bytes, 1274 zip members, 1274 with the data-descriptor flag, zip64 end record: True
   strict scan: refused -> unsafe_archive: the pickle refers to ultralytics.nn.tasks.DetectionModel
CHECK: YOLO last.pt is refused by the strict reader: True
yolo11n.pt (pretrained weights, the pinned download): 5613764 bytes, 507 zip members, 507 with the data-descriptor flag, zip64 end record: True
   strict scan: refused -> unsafe_archive: the pickle refers to ultralytics.nn.tasks.DetectionModel
adopt: CheckpointCorruptError [unsafe_archive]: the pickle refers to ultralytics.nn.tasks.DetectionModel
CHECK: adopt stored nothing for last.pt: True


**What this shows.** Ultralytics pickles the whole model object (classes and all) into `last.pt`. The
strict reader accepts only plain tensors, numbers and containers, so it refuses this file by design and
says which class it met. The same is true of the downloaded pretrained weights. CV-Pilot therefore cannot
vouch for, tamper-check or hand back an Ultralytics resume file, and `adopt` says so up front instead of
storing a file that could never be resumed. There is nothing to tamper-test: the file never gets in.

### The framework's own resume, as a control

Ultralytics resumes by itself from `last.pt` with `resume=True`. This is **not verified by CV-Pilot**; it
is here to show what the framework does with the file, and to compare the result with the uninterrupted run.

In [17]:
resume_model = ultralytics.YOLO(
    str(yolo_last)
)  # our own file, produced seconds ago by this notebook
resume_run = cvpilot.autolog(
    resume_model, project="yolo", task="detection", name="resumed-by-the-framework"
)
with timed("YOLO11n: the framework resumes"):
    resume_model.train(resume=True)
rows, ref = results("interrupted"), results("straight")
print(f"epochs after the resume: {len(rows)} (uninterrupted: {len(ref)})")
for key in ("metrics/mAP50-95(B)", "train/box_loss"):
    print(
        f"{key:24s} resumed {rows[-1][key]:.6f}  uninterrupted {ref[-1][key]:.6f}  difference {abs(rows[-1][key] - ref[-1][key]):.2e}"
    )
check("the framework's own resume ran the missing epoch", len(rows) == len(ref))
YOLO_IDENTICAL = all(
    rows[-1][k] == ref[-1][k] for k in ("metrics/mAP50-95(B)", "train/box_loss", "train/cls_loss")
)
print("final-epoch metrics identical to the uninterrupted run:", YOLO_IDENTICAL)

[YOLO11n: the framework resumes: 8.4 s]
epochs after the resume: 2 (uninterrupted: 2)
metrics/mAP50-95(B)      resumed 0.407210  uninterrupted 0.411060  difference 3.85e-03
train/box_loss           resumed 1.169630  uninterrupted 1.290970  difference 1.21e-01
CHECK: the framework's own resume ran the missing epoch: True
final-epoch metrics identical to the uninterrupted run: False


**What this shows.** The framework completes the missing epoch from its own file. The line above states
whether the final-epoch metrics are exactly equal to the uninterrupted run; this is a property of
Ultralytics on this CPU, reported as measured, and says nothing about CV-Pilot because CV-Pilot was not
in the loop for the resume.

# Part D. Hugging Face Trainer: a real checkpoint folder

`microsoft/resnet-18` (pinned commit) is fine-tuned on crops of the same photos for two epochs with
`save_strategy="epoch"`. A `Trainer` checkpoint is a **folder**: model weights, optimizer, scheduler,
random state, the arguments and a state file. CV-Pilot's verification covers single files, so this part
shows exactly which of those files the reader takes and which it does not.

In [18]:
def build_crops(side=64, per_class=40, min_side=12, classes=5):
    counts = Counter()
    rows = []
    for image in IMAGES:
        label_file = COCO_ROOT / "labels" / "train2017" / f"{image.stem}.txt"
        if not label_file.exists():
            continue
        for ln in label_file.read_text().splitlines():
            parts = ln.split()
            if len(parts) == 5:
                rows.append((image, int(parts[0]), *map(float, parts[1:])))
                counts[int(parts[0])] += 1
    keep = [c for c, _ in counts.most_common(classes)]
    taken, xs, ys = Counter(), [], []
    for image, cls, xc, yc, w, h in rows:
        if cls not in keep or taken[cls] >= per_class:
            continue
        with Image.open(image) as handle:
            handle = handle.convert("RGB")
            W, H = handle.size
            box = ((xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H)
            if box[2] - box[0] < min_side or box[3] - box[1] < min_side:
                continue
            xs.append(np.asarray(handle.crop(box).resize((side, side))))
        ys.append(keep.index(cls))
        taken[cls] += 1
    return (
        np.stack(xs).transpose(0, 3, 1, 2).astype(np.uint8),
        np.array(ys),
        [NAMES[c] for c in keep],
    )


crop_x, crop_y, crop_names = build_crops()
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)  # from the model's preprocessor config
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
CROPS = (torch.tensor(crop_x).float() / 255 - MEAN) / STD
LABELS = torch.tensor(crop_y)
print(f"{len(crop_y)} crops of {len(crop_names)} classes: {crop_names}")


class Crops(torch.utils.data.Dataset):
    def __len__(self):
        return len(LABELS)

    def __getitem__(self, i):
        return {"pixel_values": CROPS[i], "labels": LABELS[i]}


def new_trainer(folder):
    torch.manual_seed(0)
    model = transformers.AutoModelForImageClassification.from_pretrained(
        str(HF_MODEL_DIR),
        num_labels=len(crop_names),
        ignore_mismatched_sizes=True,
        use_safetensors=True,
        trust_remote_code=False,
    )
    args = transformers.TrainingArguments(
        output_dir=str(folder),
        num_train_epochs=2,
        per_device_train_batch_size=16,
        learning_rate=5e-4,
        lr_scheduler_type="constant",
        warmup_steps=0,
        weight_decay=0.0,
        logging_steps=1,
        save_strategy="epoch",
        save_total_limit=None,
        report_to=[],
        use_cpu=True,
        disable_tqdm=True,
        seed=0,
        remove_unused_columns=False,
    )
    trainer = transformers.Trainer(model=model, args=args, train_dataset=Crops())
    trainer.remove_callback(transformers.trainer_callback.PrinterCallback)  # no log dicts on stdout
    return trainer


hf_straight = new_trainer(WORK / "hf-straight")
hf_run = cvpilot.autolog(hf_straight, project="hf", task="classification", name="straight")
with timed("Hugging Face: two epochs straight"):
    hf_straight.train()
hf_view = cvpilot.runs.get_run_view("hf", hf_run.run_id)
folders = sorted(
    (WORK / "hf-straight").glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1])
)
print(
    "checkpoint folders:",
    [f.name for f in folders],
    "| CV-Pilot run:",
    hf_view.meta.status,
    f"({hf_view.stats.n_points} points)",
)
hf_losses = recorded("hf", hf_run.run_id, "train/loss")
check(
    "the hook recorded one training loss per real Trainer step and the run finished",
    hf_view.meta.status == "finished" and len(hf_losses) == hf_straight.state.global_step,
)

141 crops of 5 classes: ['person', 'car', 'cup', 'chair', 'book']

[Hugging Face: two epochs straight: 1.5 s]
checkpoint folders: ['checkpoint-9', 'checkpoint-18'] | CV-Pilot run: finished (77 points)
CHECK: the hook recorded one training loss per real Trainer step and the run finished: True


**What this shows.** A real fine-tune of a real model with the real `Trainer`, recorded by the hook.

### Which files of the folder can the reader take?

Every file of the first checkpoint folder, adopted one by one with the exact outcome. Then the folder
itself.

In [19]:
first = folders[0]
run = cvpilot.init("hf", task="classification", name="adopt-hf")
mgr = CheckpointManager(
    run, dir=WORK / "ckpt-hf", allow_pt_format=True, install_signal_handlers=False
)
step = 1
for file in sorted(first.iterdir()):
    try:
        mgr.adopt(file, step=step)
        outcome, message = "accepted", ""
        step += 1
    except Exception as exc:
        outcome, message = "refused", f"[{getattr(exc, 'reason', type(exc).__name__)}] {exc}"
    print(f"{file.name:22s} {file.stat().st_size:>10} bytes  {outcome:9s} {scrub(message)[:110]}")
    layout("Hugging Face", f"{first.name}/{file.name}", outcome, message)
try:
    mgr.adopt(first, step=step)
    outcome, message = "accepted", ""
except Exception as exc:
    outcome, message = "refused", f"{type(exc).__name__}: {exc}"
print(f"{'the folder itself':22s} {'':>10}        {outcome:9s} {scrub(message)[:110]}")
layout("Hugging Face", f"{first.name}/ (the folder)", outcome, message)
print(
    "a resume helper for this framework exists:",
    hasattr(__import__("cvpilot.hooks.hf_trainer", fromlist=["x"]), "resume_arguments"),
)
mgr.close()
cvpilot.finish("failed")
check("the Trainer folder as a whole is refused", outcome == "refused")

config.json                   882 bytes  refused   [unloadable] the file is not a zip-archive checkpoint (raw pickles and safetensors files are not adopted)
model.safetensors        44770492 bytes  refused   [unloadable] the file is not a zip-archive checkpoint (raw pickles and safetensors files are not adopted)


optimizer.pt             89483723 bytes  accepted  
rng_state.pth               14327 bytes  refused   [unsafe_archive] the pickle refers to numpy._core.multiarray._reconstruct
scheduler.pt                 1465 bytes  accepted  
trainer_state.json           2258 bytes  refused   [unloadable] the file is not a zip-archive checkpoint (raw pickles and safetensors files are not adopted)
training_args.bin            5265 bytes  refused   [unsafe_archive] the pickle refers to transformers.training_args.TrainingArguments
the folder itself                        refused   CheckpointCorruptError: checkpoint-9 is not a regular file
a resume helper for this framework exists: False
CHECK: the Trainer folder as a whole is refused: True


**What this shows.** A Trainer checkpoint is not one file, and no single file of it can resume the
Trainer. Two of its files (optimizer and scheduler state) happen to be plain enough to pass the scan;
the weights are safetensors, which `adopt` does not take (it stores framework `pt` archives); the
random-state file pickles NumPy objects and the arguments file pickles the `TrainingArguments` class, both
refused. So CV-Pilot does **not** verify or hand back a Hugging Face resume folder, and the hook offers no
`resume_arguments` for it.

### The framework's own resume, as a control

`trainer.train(resume_from_checkpoint=folder)` is the Trainer's own mechanism, again **not verified by
CV-Pilot**: here to show what it does and how it compares with the uninterrupted run.

In [20]:
hf_resumed = new_trainer(WORK / "hf-resumed")
with timed("Hugging Face: the framework resumes from epoch one"):
    hf_resumed.train(resume_from_checkpoint=str(first))
HF_STEPS = (hf_straight.state.global_step, hf_resumed.state.global_step)
HF_SAME = all(
    torch.equal(a, b)
    for a, b in zip(
        hf_straight.model.state_dict().values(), hf_resumed.model.state_dict().values(), strict=True
    )
)
print(f"global steps: uninterrupted {HF_STEPS[0]}, resumed {HF_STEPS[1]}")
print("final weights identical to the uninterrupted run:", HF_SAME)
check("the framework's own resume reached the same step", HF_STEPS[0] == HF_STEPS[1])

[Hugging Face: the framework resumes from epoch one: 0.8 s]
global steps: uninterrupted 18, resumed 18
final weights identical to the uninterrupted run: True
CHECK: the framework's own resume reached the same step: True


**What this shows.** The Trainer resumes from its own folder. Whether the final weights equal the
uninterrupted run is printed as measured; it is a property of the framework on this CPU.

### What CV-Pilot can verify for this model: its own format

`CheckpointManager.save_now` stores model and optimizer state as safetensors plus a bounded JSON
skeleton (no pickle in the file, nothing executes on restore). This is the path with full verification.
The restored state holds NumPy arrays: the cell converts them back and then continues training, once from
the restored state and once from the live objects, and compares.

In [21]:
def to_torch(value):
    if isinstance(value, np.ndarray):
        return torch.from_numpy(value.copy())
    if isinstance(value, dict):
        return {k: to_torch(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_torch(v) for v in value]
    return value


def one_step(model, optimizer, n=2):
    model.train()
    for i in range(n):
        batch = {"pixel_values": CROPS[i * 8 : i * 8 + 8], "labels": LABELS[i * 8 : i * 8 + 8]}
        loss = model(**batch).loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


live_model, live_opt = hf_straight.model, hf_straight.optimizer
run = cvpilot.init("hf", task="classification", name="native-format")
nat = CheckpointManager(run, dir=WORK / "ckpt-hf-native", install_signal_handlers=False)
with timed("save model and optimizer state with CheckpointManager"):
    saved = nat.save_now(hf_straight.state.global_step, model=live_model, optimizer=live_opt)
print(
    f"saved {saved.size_bytes} bytes ({saved.format}), restore verifies the digest before reading"
)
restored = nat.restore()
clone_trainer = new_trainer(WORK / "hf-clone")
clone_trainer.create_optimizer()  # the Trainer's own optimizer: two parameter groups (with and without decay)
clone, clone_opt = clone_trainer.model, clone_trainer.optimizer
clone.load_state_dict(to_torch(restored.state["model"]))
clone_opt.load_state_dict(to_torch(restored.state["optimizer"]))


def same_nested(a, b):
    """Equal nested state: tensors bit for bit, containers by content, numbers by value."""
    if isinstance(a, torch.Tensor) or isinstance(b, torch.Tensor):
        return isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor) and torch.equal(a, b)
    if isinstance(a, dict):
        return (
            isinstance(b, dict) and a.keys() == b.keys() and all(same_nested(a[k], b[k]) for k in a)
        )
    if isinstance(a, (list, tuple)):
        return isinstance(b, (list, tuple)) and len(a) == len(b) and all(map(same_nested, a, b))
    return a == b


check(
    "restored weights equal the saved ones",
    same_nested(live_model.state_dict(), clone.state_dict()),
)
check(
    "restored optimizer state (moments, step counters, parameter groups) equals the live optimizer's",
    same_nested(live_opt.state_dict(), clone_opt.state_dict())
    and len(clone_opt.state_dict()["state"]) > 0,
)
one_step(live_model, live_opt)
one_step(clone, clone_opt)
check(
    "two more steps from the restored model and optimizer equal two more steps from the live ones",
    all(
        torch.equal(a, b)
        for a, b in zip(live_model.state_dict().values(), clone.state_dict().values(), strict=True)
    ),
)
print("trust level reached:", restored.trust)
nat.close()
cvpilot.finish(
    "preempted"
)  # the next cells ask, from a fresh process, whether this checkpoint restores

[save model and optimizer state with CheckpointManager: 0.1 s]
saved 134252804 bytes (safetensors), restore verifies the digest before reading
CHECK: restored weights equal the saved ones: True
CHECK: restored optimizer state (moments, step counters, parameter groups) equals the live optimizer's: True


CHECK: two more steps from the restored model and optimizer equal two more steps from the live ones: True
trust level reached: anchored


In [22]:
stored = WORK / "ckpt-hf-native" / saved.run_id / saved.path
good = stored.read_bytes()
foreign_trainer = new_trainer(WORK / "hf-other")
foreign_trainer.create_optimizer_and_scheduler(
    num_training_steps=hf_straight.state.max_steps
)  # same set-up as the original
with torch.no_grad():
    for parameter in foreign_trainer.model.parameters():
        parameter.add_(0.01)  # another training run's weights
one_step(foreign_trainer.model, foreign_trainer.optimizer)
foreign_run = cvpilot.init("hf", task="classification", name="foreign")
foreign_mgr = CheckpointManager(
    foreign_run, dir=WORK / "ckpt-hf-foreign", install_signal_handlers=False
)
foreign_saved = foreign_mgr.save_now(
    saved.step, model=foreign_trainer.model, optimizer=foreign_trainer.optimizer
)
foreign_mgr.close()
cvpilot.finish("preempted")
other_bytes = (WORK / "ckpt-hf-foreign" / foreign_saved.run_id / foreign_saved.path).read_bytes()
check(
    "the foreign checkpoint is valid, has the same size and other content",
    len(other_bytes) == len(good) and other_bytes != good and foreign_saved.sha256 != saved.sha256,
)
NATIVE_ROWS, NATIVE_EVENTS = [], []
probe_env = {"PROBE_PARENT": saved.run_id, "PROBE_DIR": str(WORK / "ckpt-hf-native")}
for name, bad in {
    "flip one byte": good[:5000] + bytes([good[5000] ^ 1]) + good[5001:],
    "truncate to half": good[: len(good) // 2],
    "swap in a foreign safetensors file of the same size": other_bytes,
}.items():
    stored.chmod(0o600)
    stored.write_bytes(bad)
    shown, events = run_probe("probe_restore.py", probe_env)
    NATIVE_ROWS.append((name, "a file was returned" not in shown))
    NATIVE_EVENTS.append(" ".join(events))
    print(f"{name:36s} restore in a fresh process -> {shown}")
    print(f"{'':36s} warnings logged: {', '.join(dict.fromkeys(events))}")
stored.write_bytes(good)
stored.chmod(0o444)
check(
    "every tampered safetensors checkpoint was refused", all(refused for _, refused in NATIVE_ROWS)
)
check(
    "the same-size foreign file was caught by the digest, not by its size",
    any("digest_mismatch" in row for row in NATIVE_EVENTS[-1:]),
)
check(
    "the untouched file restores again in a fresh process",
    run_probe("probe_restore.py", probe_env)[0] == "a file was returned",
)
layout(
    "CheckpointManager (safetensors)",
    "model + AdamW state of the Hugging Face model",
    "accepted",
    "",
)

CHECK: the foreign checkpoint is valid, has the same size and other content: True


flip one byte                        restore in a fresh process -> None (nothing handed back)
                                     warnings logged: checkpoint_skipped_corrupt(digest_mismatch), checkpoint_progress_lost(rolled_back)


truncate to half                     restore in a fresh process -> None (nothing handed back)
                                     warnings logged: checkpoint_skipped_corrupt(size_mismatch), checkpoint_progress_lost(rolled_back)


swap in a foreign safetensors file of the same size restore in a fresh process -> None (nothing handed back)
                                     warnings logged: checkpoint_skipped_corrupt(digest_mismatch), checkpoint_progress_lost(rolled_back)
CHECK: every tampered safetensors checkpoint was refused: True
CHECK: the same-size foreign file was caught by the digest, not by its size: True


CHECK: the untouched file restores again in a fresh process: True


**What this shows.** The path with full verification works on a real pretrained model with a real optimizer:
the restored weights and the optimizer state are bit-identical, two further steps equal the live ones, and
a flipped byte, a truncation and a foreign file are all refused. This path never needed the pickle scan.

# Part E. MMEngine: a real run with the hook, and its real checkpoint

The MMEngine hook records training scalars and validation and test metrics and marks the run finished or
failed. The tiny model trains on the same thumbnails as Part B.

In [23]:
from mmengine.evaluator import BaseMetric
from mmengine.model import BaseModel
from mmengine.registry import METRICS
from mmengine.runner import Runner


class MMNet(BaseModel):
    def __init__(self):
        super().__init__()
        self.layer = torch.nn.Linear(X.shape[1], len(frequent))

    def forward(self, inputs, data_samples=None, mode="tensor"):
        logits = self.layer(inputs)
        if mode == "loss":
            return {"loss": torch.nn.functional.cross_entropy(logits, data_samples)}
        if mode == "predict":
            return list(logits.argmax(1))
        return logits


class MMData(torch.utils.data.Dataset):
    def __len__(self):
        return len(Y)

    def __getitem__(self, i):
        return {"inputs": torch.tensor(X[i]), "data_samples": torch.tensor(Y[i])}


@METRICS.register_module(force=True)
class Accuracy(BaseMetric):
    default_prefix = "toy"

    def process(self, data_batch, data_samples):
        labels = data_batch["data_samples"]
        self.results.append(float((torch.stack(data_samples).cpu() == labels.cpu()).float().mean()))

    def compute_metrics(self, results):
        return {"accuracy": sum(results) / len(results)}


def mm_loader():
    return torch.utils.data.DataLoader(MMData(), batch_size=16)


mm_runner = Runner(
    model=MMNet(),
    work_dir=str(WORK / "mm"),
    train_dataloader=mm_loader(),
    val_dataloader=mm_loader(),
    test_dataloader=mm_loader(),
    val_evaluator=dict(type="Accuracy"),
    test_evaluator=dict(type="Accuracy"),
    val_cfg={},
    test_cfg={},
    train_cfg={"by_epoch": True, "max_epochs": 2, "val_interval": 1},
    optim_wrapper={"optimizer": {"type": "SGD", "lr": 0.5}},
    default_hooks={"checkpoint": {"type": "CheckpointHook", "interval": 1}},
    log_level="ERROR",
)
from cvpilot.hooks.mmengine import MMEngineHook

mm_run = cvpilot.init("mm", task="classification", name="tiny")
MMEngineHook(interval=1).attach(mm_runner, mm_run)
with timed("MMEngine: train, validate, test"):
    mm_runner.train()
    during = cvpilot.runs.get_run_view("mm", mm_run.run_id).meta.status
    mm_runner.test()
view = cvpilot.runs.get_run_view("mm", mm_run.run_id)

store = open_store("mm", Path("store"))
points = store.read_points(mm_run.run_id)
store.close()
keys = sorted({p.key for p in points})
print(
    f"status after train(): {during}; after test(): {view.meta.status}; {len(points)} points under {keys}"
)
check(
    "the run stayed open after train and finished when test returned",
    during == "running" and view.meta.status == "finished",
)
check(
    "training, validation and test scalars were all recorded",
    {"train/loss", "val/toy/accuracy", "test/toy/accuracy"} <= set(keys),
)
steps = [p.step for p in points if p.key == "train/loss"]
check("steps are strictly increasing", steps == sorted(set(steps)) and len(steps) > 1)

[MMEngine: train, validate, test: 0.3 s]
status after train(): running; after test(): finished; 11 points under ['test/toy/accuracy', 'train/loss', 'val/toy/accuracy']
CHECK: the run stayed open after train and finished when test returned: True
CHECK: training, validation and test scalars were all recorded: True
CHECK: steps are strictly increasing: True


In [24]:
mm_files = sorted((WORK / "mm").glob("epoch_*.pth"))
print("checkpoint files written by the framework:", [f.name for f in mm_files])
check(
    "the MMEngine .pth is refused by the strict reader",
    show("MMEngine", mm_files[-1].name, mm_files[-1]) == "refused",
)

checkpoint files written by the framework: ['epoch_1.pth', 'epoch_2.pth']
epoch_2.pth: 20517 bytes, 8 zip members, 8 with the data-descriptor flag, zip64 end record: True
   strict scan: refused -> unsafe_archive: the pickle refers to mmengine.logging.history_buffer.HistoryBuffer
CHECK: the MMEngine .pth is refused by the strict reader: True


**What this shows.** The hook works against the real runner: scalars and metrics were recorded in step order
and the run's lifecycle follows `train` and `test`. The framework's checkpoint pickles its logging buffers by
class, so the strict reader refuses it for the same reason as the Ultralytics file; the MMEngine hook
therefore has no resume helper, and a resume stays the framework's own, unverified.

# Part F. What a save costs on real model sizes, and where the time goes

`benchmarks/ckpt_save.py` times a steady-state `CheckpointManager.save_now` (retention on, the cost a long
job pays), splits it into its phases, and compares it with the framework's own `torch.save` of the same
state, written and `fsync`ed to the same disk. The three states are the real ones from this notebook.

In [25]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    "ckpt_save_bench", REPO / "benchmarks" / "ckpt_save.py"
)
bench = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bench)


def torch_baseline(state):
    return lambda path: torch.save(state, path)


tiny_state = {k: v.detach() for k, v in half.state_dict().items()}
resnet_state = {k: v.detach() for k, v in live_model.state_dict().items()}
yolo_state = {
    k: v.detach().float() for k, v in ultralytics.YOLO(str(weights_path)).model.state_dict().items()
}
SIZES = {
    "Lightning tiny classifier": tiny_state,
    "YOLO11n weights": yolo_state,
    "ResNet-18 weights": resnet_state,
}
MEASURED = {}
for label, state in SIZES.items():
    MEASURED[label] = bench.profile_state(state, baseline=torch_baseline(state), repeats=9)
print(
    f"{'state':28s} {'bytes':>11} {'save_now':>10} {'torch.save':>11} {'ratio':>6} {'sha256 pass':>12} {'ms per MB':>10}"
)
for label, m in MEASURED.items():
    print(
        f"{label:28s} {m['bytes']:>11} {1000 * m['save_now_s']:>8.2f}ms {1000 * m['framework_save_fsync_s']:>9.2f}ms "
        f"{m['save_now_s'] / m['framework_save_fsync_s']:>5.1f}x {1000 * m['sha256_pass_s']:>10.2f}ms {1000 * m['save_now_s'] / (m['bytes'] / 1e6):>10.2f}"
    )

state                              bytes   save_now  torch.save  ratio  sha256 pass  ms per MB
Lightning tiny classifier          15692     4.48ms      0.44ms  10.2x       0.01ms     285.78
YOLO11n weights                 10653476    19.34ms     12.05ms   1.6x       3.49ms       1.82
ResNet-18 weights               44790276    52.71ms     16.67ms   3.2x      14.77ms       1.18


In [26]:
print("where a steady-state save spends its time (share of save_now):")
DOMINANT = {}
for label, m in MEASURED.items():
    phases = sorted(m["phases_s"].items(), key=lambda kv: -kv[1])
    DOMINANT[label] = phases[0][0]
    print(f"\n{label}:")
    for name, seconds in phases[:6]:
        print(f"  {name:55s} {1000 * seconds:8.2f} ms  {100 * seconds / m['save_now_s']:5.1f} %")
BUDGET = 0.05
for label, m in MEASURED.items():
    over = m["save_now_s"] / m["framework_save_fsync_s"] - 1
    m["overhead_over_framework_save"] = over
    print(
        f"VERDICT: {label}: save_now costs {100 * over:.0f} % more than torch.save of the same state; the 5 % budget is "
        f"{'met' if over <= BUDGET else 'not met'}"
    )
for label, m in MEASURED.items():
    hashing = 2 * m["sha256_pass_s"] / m["save_now_s"]
    print(
        f"VERDICT: {label}: the two SHA-256 passes of a save (while writing, and when the file is read back) are "
        f"{100 * hashing:.0f} % of save_now"
    )
for label, m in MEASURED.items():
    explained = {k: v for k, v in m["phases_s"].items() if not k.startswith("unattributed")}
    unattributed = m["phases_s"][[k for k in m["phases_s"] if k.startswith("unattributed")][0]]
    print(
        f"unattributed part of a save, {label}: {1000 * unattributed:.2f} ms ({100 * unattributed / m['save_now_s']:.1f} %)"
    )
    if label != "Lightning tiny classifier":
        check(
            f"{label}: no phase time is negative and the unattributed part is under a quarter of the save",
            all(v >= 0 for v in explained.values()) and abs(unattributed) < 0.25 * m["save_now_s"],
        )
interval = ckpt_manager.DEFAULT_INTERVAL_MINUTES
for label, m in MEASURED.items():
    share = m["save_now_s"] / (interval * 60)
    print(
        f"VERDICT: {label}: one save every {interval:g} minutes (the default) is {100 * share:.4f} % of the wall time"
    )

where a steady-state save spends its time (share of save_now):

Lightning tiny classifier:
  unattributed (state build, locking, scans, timer cost)      1.91 ms   42.6 %
  retention                                                   1.40 ms   31.2 %
  pointer write                                               0.46 ms   10.3 %
  data write (hash + write + fsync)                           0.19 ms    4.2 %
  verification re-read (hash)                                 0.19 ms    4.2 %
  manifest write                                              0.16 ms    3.6 %

YOLO11n weights:
  data write (hash + write + fsync)                           5.53 ms   28.6 %
  verification re-read (hash)                                 4.27 ms   22.1 %
  serialise                                                   3.64 ms   18.8 %
  unattributed (state build, locking, scans, timer cost)      2.64 ms   13.7 %
  retention                                                   1.98 ms   10.2 %
  pointer write       

**What this shows.** The ratio column is the honest comparison with the framework's own save of the same
state on the same disk; the phase breakdown says which step dominates. The numbers are for this machine only
(note that `os.fsync` on macOS flushes to the drive's cache, not to the platter, so the fsync share is lower
here than it would be on a Linux server with a journaling filesystem). A statement about a documented
budget is only made in the VERDICT lines, from these numbers.

# Summary: every real file layout, and what the reader said

The table is built from the outcomes recorded above, with the exact message the reader gave for every
refusal.

In [27]:
print(f"{'framework':32s} {'file':52s} {'outcome':9s} message")
for framework, file, outcome, message in LAYOUTS:
    print(f"{framework:32s} {file[:52]:52s} {outcome:9s} {message[:150]}")
print()
failed = [label for label, ok in CHECKS if not ok]
print(f"checks: {len(CHECKS)}, failed: {len(failed)}", failed)
print("timings (s):", TIMINGS)

framework                        file                                                 outcome   message
Lightning                        last.ckpt                                            accepted  
synthetic hostile                model object saved by the framework                  refused   CheckpointCorruptError[unsafe_archive]: the pickle refers to torch.nn.modules.linear.Linear
synthetic hostile                os.system call saved by the framework                refused   CheckpointCorruptError[unsafe_archive]: the pickle refers to posix.system
Ultralytics                      last.pt (training state, written by train)           refused   unsafe_archive: the pickle refers to ultralytics.nn.tasks.DetectionModel
Ultralytics                      yolo11n.pt (pretrained weights, the pinned download) refused   unsafe_archive: the pickle refers to ultralytics.nn.tasks.DetectionModel
Hugging Face                     checkpoint-9/config.json                             refused   [unload

**What this shows.** Which real files are accepted (and so verified, tamper-checked and resumable) and
which are refused, with the reader's own words. Anything in the refused rows is outside what CV-Pilot
verifies: the notebook states this rather than working around it.